In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent / "src"))
import config
import pandas as pd
import numpy as np
import datetime

The purpose of this notebook is to engineer features for a churn model. As most powerlifting federation memberships operate on a calendar-year basis, predictions are made at the end of each year. The dataset is therefore structured as panel data, where each record represents a unique combination of lifter and year. Features are constructed using only information available up to the end of that year.

In [2]:
cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_comp_hist.csv")
cleaned['Date'] = pd.to_datetime(cleaned['Date'], format = '%Y-%m-%d')

In [3]:
df_sorted = cleaned.sort_values(['Name', 'Year', 'Date'])

# time between end of year and first competition (days)
df_sorted['TimeCompetingYearEnd'] = pd.to_datetime(df_sorted['Year'].astype(str) + '-12-31') - df_sorted.groupby('Name')['Date'].transform('min')
df_sorted['TimeCompetingYearEnd'] = df_sorted['TimeCompetingYearEnd'].dt.days

#number of meets competed in that year
df_sorted['NumberOfMeets'] = df_sorted.groupby(['Name', 'Year']).transform('size')

#time between first competition and that particular comp. used for computing other features
df_sorted['TimeCompeting'] = (df_sorted['Date'] - df_sorted.groupby('Name')['Date'].transform('min')).dt.days

### Rate of improvement features

It might be expected that lifters that are making progress are more likely to continue competing. Therefore a set of features relating to the rate of progress are engineered below.

In [4]:
# BestTotalYear used to compute improvement graident between years, which is computed after transformation to panel data
#Gives the best total of the year and the date that the time that lifter had been competing for when they achieved their best total
grouped = df_sorted.groupby(['Name', 'Year'])
df_sorted['BestTotalYear'] = grouped['TotalKg'].transform('max')
idx = grouped['TotalKg'].idxmax()
idx = idx.dropna()
time_best = df_sorted.loc[idx, ['Name', 'Year', 'TimeCompeting']]
time_best = time_best.rename(columns={'TimeCompeting': 'TimeCompetingBestTotalYear'})
df_sorted = df_sorted.merge(time_best, on = ['Name', 'Year'], how = 'left')

df_sorted['ImprovementGradientWithinYear'] = (
    (grouped['TotalKg'].transform('last') -
     grouped['TotalKg'].transform('first')) /
    (grouped['TimeCompeting'].transform('last') -
     grouped['TimeCompeting'].transform('first')).replace(0,np.nan)
)

df_sorted['PercentageImprovementGradientWithinYear'] = (
    df_sorted['ImprovementGradientWithinYear'] /
    (grouped['TotalKg'].transform('first')).replace(0,np.nan)
)

df_sorted['PrevMeetTotal'] = df_sorted.groupby('Name').shift(1)['TotalKg']
df_sorted['PrevMeetTimeCompeting'] = df_sorted.groupby('Name').shift(1)['TimeCompeting']

df_sorted['ImprovementGradientTwoMeets'] = (
    (df_sorted['TotalKg'] - df_sorted['PrevMeetTotal'])/
    (df_sorted['TimeCompeting']- df_sorted['PrevMeetTimeCompeting']).replace(0,np.nan)
)

df_sorted['PercentageImprovementGradientTwoMeets'] = df_sorted['ImprovementGradientTwoMeets']/df_sorted['PrevMeetTotal']

df_sorted['ImprovementAcceleration'] = (
    df_sorted.groupby('Name')['ImprovementGradientTwoMeets'].diff()/
    (df_sorted['TimeCompeting']- df_sorted['PrevMeetTimeCompeting']).replace(0,np.nan)
)

df_sorted['ImprovementAccelerationForPercentage'] = (
    df_sorted.groupby('Name')['PercentageImprovementGradientTwoMeets'].diff()/
    (df_sorted['TimeCompeting']- df_sorted['PrevMeetTimeCompeting']).replace(0,np.nan)
)

df_sorted = df_sorted.drop(columns = ['PrevMeetTotal', 'PrevMeetTimeCompeting'])



#used to get calculate improvement between years after data has been transformed to panel data
df_sorted['BestMeetOfYear'] = df_sorted.groupby(['Name', 'Year'])['TotalKg'].transform('max')

### Encoding federation 

Treating federation as a categorical feature was not practical due to issues such as name changes and federations splitting. Instead, the proportion of lifters competing under each federation was used. Each lifter was assigned to a federation based on the one they competed under most frequently within that calendar year.

In [ ]:
df_sorted['PrimaryFederation'] = (df_sorted.groupby(['Name', 'Year'])['Federation']).transform(lambda x: x.value_counts().index[0])


fed_counts = (
    df_sorted[['Name', 'Year', 'PrimaryFederation']]
    .drop_duplicates()
    .groupby(['Year', 'PrimaryFederation'])
    .size()
    .reset_index(name='FederationCount')
)

# Total number of lifters per year
year_totals = (
    df_sorted[['Name', 'Year']]
    .drop_duplicates()
    .groupby('Year')
    .size()
    .reset_index(name='YearTotal')
)

fed_prop = fed_counts.merge(year_totals, on='Year', how='left')

fed_prop['FederationProportion'] = (
    fed_prop['FederationCount'] / fed_prop['YearTotal']
)

# Merge back
df_sorted = df_sorted.merge(
    fed_prop[['Year', 'PrimaryFederation', 'FederationProportion', 'FederationCount']],
    on=['Year', 'PrimaryFederation'],
    how='left'
)

### Time since last pb

In [ ]:
#Time since last PB at the end of the relevant year.

df_sorted['PB'] = df_sorted.groupby('Name')['TotalKg'].cummax()
df_sorted['PBBeforeMeet'] = df_sorted.groupby('Name')['PB'].shift(1)
df_sorted['IsPB'] = df_sorted['TotalKg']> df_sorted['PBBeforeMeet']
#first meet should count as PB
df_sorted['IsPB'] = df_sorted['PBBeforeMeet'].isna() | df_sorted['IsPB']
df_sorted['PBTimeCompeting'] = df_sorted['TimeCompeting'].where(df_sorted['IsPB'])
df_sorted['LastPBTimeCompeting'] = df_sorted.groupby('Name')['PBTimeCompeting'].ffill()
df_sorted['TimeSinceLastPBYearEnd'] = df_sorted['TimeCompetingYearEnd'] - df_sorted['LastPBTimeCompeting']

df_sorted = df_sorted.drop(columns = ['PB','PBBeforeMeet', 'IsPB', 'PBTimeCompeting', 'LastPBTimeCompeting'])

### Features related to number of attempts made per meet

Powerlifting meets consist of 9 'attempts' at lifting weights chosen by the lifter (similar to how high jumpers choose the height of the bar). It is reasonable to suggest that the number of successful attempts a lifter makes may carry information about how likely they are to continue competing. 

It is also worth noting that a lifter has to make at least one successful attempt for each of squat, bench and deadlift in order to register a total for the competition. Otherwise they "bomb out" of the competition.

A lifter was flagged as having bombed out if they failed to record a valid lift in any discipline (squat, bench, or deadlift). For each lift, this was defined as all three attempts and the best lift being either negative (unsucessful attempts are often recorded as negative) or missing. The final BombOut flag is set to true if this condition is met for at least one discipline.

In [ ]:
bomb_squat = (
    ((df_sorted['Squat1Kg'] <= 0) | df_sorted['Squat1Kg'].isna()) &
    ((df_sorted['Squat2Kg'] <= 0) | df_sorted['Squat2Kg'].isna()) &
    ((df_sorted['Squat3Kg'] <= 0) | df_sorted['Squat3Kg'].isna()) & 
    ((df_sorted['Best3SquatKg'] <= 0) | df_sorted['Best3SquatKg'].isna())
)

bomb_bench = (
    ((df_sorted['Bench1Kg'] <= 0) | df_sorted['Bench1Kg'].isna()) &
    ((df_sorted['Bench2Kg'] <= 0) | df_sorted['Bench2Kg'].isna()) &
    ((df_sorted['Bench3Kg'] <= 0) | df_sorted['Bench3Kg'].isna()) & 
    ((df_sorted['Best3BenchKg'] <= 0) | df_sorted['Best3BenchKg'].isna())
)

bomb_deadlift = (
    ((df_sorted['Deadlift1Kg'] <= 0) | df_sorted['Deadlift1Kg'].isna()) &
    ((df_sorted['Deadlift2Kg'] <= 0) | df_sorted['Deadlift2Kg'].isna()) &
    ((df_sorted['Deadlift3Kg'] <= 0) | df_sorted['Deadlift3Kg'].isna()) & 
    ((df_sorted['Best3DeadliftKg'] <= 0) | df_sorted['Best3DeadliftKg'].isna())
)


df_sorted['BombOut'] = bomb_squat | bomb_bench | bomb_deadlift

Number of meets since last bombout, capped at 999. If lifter has never bombed out this is set to the maximum cap of 999.

In [ ]:
#later, the last row of each year is taken to form the panrl data. 
# i.e. number of career bombouts at the year end is feature. Meets since last bombout at year end is a feature.
df_sorted['NumberOfBombOuts'] = df_sorted.groupby('Name')['BombOut'].cumsum()
df_sorted['PercentageBombOuts'] = df_sorted['NumberOfBombOuts']/(df_sorted.groupby('Name').cumcount()+1)
df_sorted['MeetsSinceLastBombOut'] = df_sorted.groupby(['Name', 'NumberOfBombOuts']).cumcount()

#past capped number of bombouts it gets set to cap
df_sorted.loc[df_sorted['MeetsSinceLastBombOut']> config.CAP_MEETS_SINCE_BOMBOUT, 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT

#if lifter has never bombed out number of meets since last bombout is set to cap
df_sorted.loc[(df_sorted['NumberOfBombOuts'] ==0 ), 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT



Calculating the number of attempts made by the lifter at the meet. If the individual attempt fields were missing but the lifter did not bomb out (for example, only Best3SquatKg, Best3BenchKg, and Best3DeadliftKg were recorded), AttemptsMade was imputed using the mean number of attempts made by lifters who did not bomb out.

In [ ]:
attempt_cols = ['Squat1Kg','Squat2Kg','Squat3Kg',
                   'Bench1Kg','Bench2Kg','Bench3Kg',
                   'Deadlift1Kg','Deadlift2Kg','Deadlift3Kg']

df_sorted['AttemptsMade'] = (df_sorted[attempt_cols]>0).sum(axis=1)

#handling the case where the lifter didnt bombout but it's recorded as 0 attempts because individual attempst weren't filled in 
mask = (df_sorted['BombOut']==False)&(df_sorted['AttemptsMade']==0)

#excluding lifters who bombed out from calculation as we already know this lifter did not bomb out
mean_attempts = df_sorted.loc[(df_sorted['BombOut']==False) & (df_sorted['AttemptsMade']>0),'AttemptsMade'].mean().round() 

df_sorted.loc[mask, 'AttemptsMade'] = mean_attempts

In [ ]:
grouped = df_sorted.groupby(['Name', 'Year'])['AttemptsMade']

df_sorted['LastMeetAttemptsMade'] = grouped.transform('last')
df_sorted['AverageAttemptsMade'] = grouped.transform('mean')
df_sorted['MostAttemptsMade'] = grouped.transform('max')
df_sorted['LeastAttemptsMade'] = grouped.transform('min')




Next the data was transformed to panel data. Some additional features were computed after this. 

In [ ]:
panel_data = df_sorted.groupby(['Name', 'Year']).tail(1).reset_index(drop = True)
panel_data = panel_data.sort_values(['Name', 'Year'])

panel_data['BestTotalPrevYear'] = panel_data.groupby('Name')['BestTotalYear'].shift(1)
panel_data['TimeCompetingBestTotalPrevYear'] = panel_data.groupby('Name')['TimeCompetingBestTotalYear'].shift(1)

panel_data['ImprovementGradientBetweenYears'] = (
    (panel_data['BestTotalYear'] - panel_data['BestTotalPrevYear'])/
    (panel_data['TimeCompetingBestTotalYear'] - panel_data['TimeCompetingBestTotalPrevYear']).replace(0, np.nan)
)
panel_data['PercentageImprovementGradientBetweenYears']= (
    panel_data['ImprovementGradientBetweenYears']/ (panel_data['BestTotalPrevYear']).replace(0, np.nan)
)

panel_data['MeetsYoYChange'] = panel_data.groupby('Name')['NumberOfMeets'].diff()

panel_data['AvgMeetsPerYear'] = (
    panel_data.groupby('Name')['NumberOfMeets']
    .expanding().mean()
    .reset_index(level=0, drop=True)
)
panel_data['BelowAvgMeets'] = (
    panel_data['NumberOfMeets'] < panel_data['AvgMeetsPerYear']
).astype(int)

In [ ]:
cols =[
    'Name','Year', 'Date', 'Sex','WeightClassKg','Age', 'TimeCompetingYearEnd', 'NumberOfMeets',
    'Goodlift',  'AverageAttemptsMade', 'MostAttemptsMade',
    'LeastAttemptsMade','LastMeetAttemptsMade',
    'MeetsSinceLastBombOut', 'NumberOfBombOuts',
    'PercentageImprovementGradientWithinYear', 'ImprovementGradientWithinYear',
    'PercentageImprovementGradientTwoMeets', 'ImprovementGradientTwoMeets',
    'PercentageImprovementGradientBetweenYears', 'ImprovementGradientBetweenYears',
    'TimeSinceLastPBYearEnd',
    'PercentageBombOuts',
    'ImprovementAccelerationForPercentage', 'ImprovementAcceleration',
    'FederationProportion', 
    'MeetsYoYChange', 'AvgMeetsPerYear','BelowAvgMeets'
]
panel_data = panel_data.loc[:, cols]

#target column is Churns
panel_data['CompetesNextYear'] = (
    (panel_data['Name'] == panel_data['Name'].shift(-1)) & 
    (panel_data['Year'] +1 == panel_data['Year'].shift(-1))
)
panel_data['Churns'] = (~panel_data['CompetesNextYear']).astype(int)
panel_data = panel_data.drop(columns = 'CompetesNextYear')

#Need to drop the entries with the year as 2025 because we cannot know whether someone competed in 2026 yet.
max_year = panel_data['Year'].max()
panel_data = panel_data[panel_data['Year'] < max_year]

#### Encoding weight class

Weight classes were encoded as (weight class rank)/(number of classes), such that the heaviest weight class present in a given year for a given sex is 1 and the lightest weight class present in that year is 0.


Weight classes were scaled by rank within each year and sex, with the lightest class mapped to 0 and the heaviest to 1

In [ ]:
#ensures 84+ and 120+ classes are recognised
def w_cl_to_num(w):
    w = str(w)
    return float(w[:-1]) + 0.5 if w.endswith('+') else float(w)

panel_data['WeightClassKgNum'] = panel_data['WeightClassKg'].apply(w_cl_to_num)

panel_data['WeightClassKgCode'] = (
    panel_data.groupby(['Sex', 'Year'])['WeightClassKgNum']
    .transform(lambda x: (x.rank(method='dense')-1)/(x.nunique()-1))
)

panel_data = panel_data.drop(columns=['WeightClassKgNum', 'WeightClassKg'])

#encoding Sex. Will use HistGradientBoostingClassifier which can handle categorical features.
panel_data['Sex'] = panel_data['Sex'].map({'F': 0, 'M': 1, 'Mx': 2})

#feature to flag Covid
panel_data['CovidAffectedPrediction'] = (
    panel_data['Year'].isin([2019])
).astype(int)

panel_data = panel_data.sort_values('Date', ascending= True)
#
panel_data = panel_data.drop(columns = 'Date')


In [ ]:
panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data.to_csv(panel_data_path, index = False)